# 🧠 The Universal SQL Problem-Solving Framework: Complete Substep Breakdown

Welcome to the **Universal SQL Problem-Solving Framework**.

Whether you are tackling entry-level aggregations or Staff-level streaming window analytics, complex SQL should **never be written all at once**. Instead, solve every case backwards using the **4-Step Modular CTE Architecture**.

---

Always work backwards from what the stakeholder wants to see:

<pre style="background: transparent !important; background-color: transparent !important; border: none !important; font-family: 'Courier New', Courier, monospace; font-size: 13px; line-height: 1.25; color: inherit; padding: 0; margin: 15px 0;">
┌────────────────────────────────────────────────────────┐
│ STEP 1: TARGET OUTPUT (The Destination)                │
│ • What columns and metrics are required?               │
│ • What does exactly 1 row represent (Target Grain)?    │
│ • What is the final presentation? (Top-N / Ordering)   │
└──────────────────────────┬─────────────────────────────┘
                           ▼
┌────────────────────────────────────────────────────────┐
│ STEP 2: RAW SCHEMA AUDIT (The Starting Point)          │
│ • Which tables contain the raw columns?                │
│ • Identify quirks: Nulls, string types, join keys      │
│ • Audit join cardinality: Avoid 1:N row duplication    │
└──────────────────────────┬─────────────────────────────┘
                           ▼
┌────────────────────────────────────────────────────────┐
│ STEP 3: BACKWARD CTE TRANSFORMATION PIPELINE (Bridge)  │
│ • CTE 1: Cast Types ➔ Filter Early ➔ Strip Dates       │
│ • CTE 2: Pre-Aggregate Secondary Tables to 1:1 Grain   │
│ • CTE 3: Join Tables ➔ Window Functions ➔ Row Math     │
│ • Final SELECT: Group by Target Grain ➔ Aggregate      │
└──────────────────────────┬─────────────────────────────┘
                           ▼
┌────────────────────────────────────────────────────────┐
│ STEP 4: SANITY AUDIT & VERIFICATION (The Guarantee)    │
│ • Guard against 0-division using NULLIF(col, 0)        │
│ • Prevent NULL propagation with COALESCE(col, default) │
│ • Assert Grain Uniqueness with COUNT(*) vs COUNT(DIST) │
└────────────────────────────────────────────────────────┘
</pre>

---

# 🎯 STEP 1: TARGET OUTPUT (The Destination)
> **Core Idea:** You cannot write a query if you don't know the exact shape and grain of the final deliverable.

### 🔹 Substep 1.1: Target Schema (Required Columns & Metrics)
* **What it means:** List the exact names and types of columns required in the final `SELECT` clause:
  - **Dimensions (Group Keys):** `account_tier`, `region`
  - **Aggregated Metrics:** `total_volume_usd`, `fraud_volume_usd`, `active_customers`
  - **Derived Ratios:** `fraud_rate_pct` (`(fraud_volume / total_volume) * 100`)
* **Why it matters:** Eliminates confusion on what to calculate and prevents writing unnecessary subqueries.

---

### 🔹 Substep 1.2: Target Grain (Level of Detail)
* **What it means:** Ask yourself: *"What does **exactly ONE row** in the final output represent?"*
* **Examples:**
  - If 1 row = 1 Customer -> `GROUP BY customer_id`
  - If 1 row = 1 Region x Account Tier -> `GROUP BY region, account_tier`
  - If 1 row = 1 Date -> `GROUP BY DATE(transaction_date)`
* **Why it matters:** **Grain is the #1 concept in SQL.** If you perform a join without aligning grain, your metrics will multiply exponentially (e.g., 5x duplicated volume).

---

### 🔹 Substep 1.3: Target Presentation (Ordering & Top-N)
* **What it means:** Determine how the output should be sorted and truncated:
  - `ORDER BY total_volume_usd DESC`
  - `LIMIT 3` (or pagination with `OFFSET`)

---
---

# 🔍 STEP 2: RAW SCHEMA AUDIT (The Starting Point)
> **Core Idea:** Inspect your tables and join relationships before writing a single line of business logic.

### 🔹 Substep 2.1: Source Table Mapping
* **What it means:** Map each required field back to its raw table:
  - `transaction_amount`, `is_fraud`, `transaction_date` -> in `transactions`
  - `account_tier`, `kyc_status`, `current_balance` -> in `customers`
  - `spot_rate`, `bid_rate`, `ask_rate` -> in `fx_rates`
* **Decision:** Identify the **Primary Fact Table** (highest cardinality, e.g. `transactions`) vs **Dimension Tables** (e.g. `customers`).

---

### 🔹 Substep 2.2: Data Types & String Quirks
* **What to check:**
  1. **Numeric values stored as strings:** `'100.50'` -> must wrap with `CAST(amount AS REAL)`.
  2. **Timestamps vs Dates:** `'2025-01-01 14:30:00'` will fail equality with `'2025-01-01'` -> must wrap with `DATE(transaction_date)`.
  3. **Case and Whitespace:** `'Completed '` vs `'completed'` -> must sanitize with `LOWER(TRIM(status))`.

---

### 🔹 Substep 2.3: Join Keys & 1:N Cardinality Check
* **The #1 SQL Interview Bug:** If table A has 10 transactions on day 1, and table B has 5 rate records on day 1, doing `FROM A JOIN B ON A.date = B.date` explodes the output to `10 * 5 = 50` rows!
* **Golden Rule:** **Pre-aggregate the secondary table in a CTE to 1 row per join key BEFORE joining.**

---
---

# 🛠️ STEP 3: BACKWARD CTE TRANSFORMATION PIPELINE (The Bridge)
> **Core Idea:** Build the query modularly using Common Table Expressions (CTEs).

Follow this exact 4-Block CTE execution recipe:

**`CTE 1: Filter & Cast ➔ CTE 2: Pre-Aggregate Secondary ➔ CTE 3: Join & Window Math ➔ Final SELECT: Group & Rank`**

In [ ]:
# Setup in-memory SQLite Engine & %%sql Studio
import sqlite3
import pandas as pd
import os
from IPython import get_ipython
from IPython.core.magic import register_line_cell_magic

conn = sqlite3.connect(':memory:')

def load_table(name, path, sep=','):
    if os.path.exists(path):
        df = pd.read_csv(path, sep=sep)
        df.to_sql(name, conn, index=False, if_exists='replace')

base_dir = 'data' if os.path.exists('data') else '../data'
load_table('transactions', os.path.join(base_dir, 'raw_transactions.csv'))
load_table('customers', os.path.join(base_dir, 'customers.csv'))
load_table('merchants', os.path.join(base_dir, 'merchants.csv'))
load_table('disputes', os.path.join(base_dir, 'disputes.csv'))
load_table('fx_rates', os.path.join(base_dir, 'fx_rates_daily.tsv'), sep='\t')

def _execute_raw_sql(query):
    query = query.strip()
    if query.upper().startswith(('INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'VACUUM', 'ANALYZE', 'BEGIN', 'COMMIT', 'ROLLBACK')):
        cur = conn.cursor()
        cur.executescript(query)
        conn.commit()
        return 'Query Executed Successfully.'
    else:
        return pd.read_sql_query(query, conn)

@register_line_cell_magic
def sql(line, cell=None):
    return _execute_raw_sql(cell if cell is not None else line)

print('🚀 SQL Studio Active! Ready for %%sql queries.')

### 🔹 Substep 3.1 & 3.2: Filter Early & Type-Cast (CTE 1)
Clean the primary fact table, sanitize strings, strip dates, and filter out bad rows early:

In [ ]:
%%sql
-- CTE 1: Clean transactions fact table
WITH valid_tx AS (
    SELECT 
        CAST(transaction_id AS INTEGER) AS transaction_id,
        CAST(customer_id AS INTEGER) AS customer_id,
        DATE(transaction_date) AS tx_date,
        CAST(transaction_amount AS REAL) AS tx_amount,
        CAST(is_fraud AS INTEGER) AS is_fraud,
        TRIM(region) AS region
    FROM transactions
    WHERE LOWER(TRIM(transaction_status)) = 'completed'
)
SELECT * FROM valid_tx LIMIT 3;

### 🔹 Substep 3.3: Pre-Aggregate Secondary Table (CTE 2)
Aggregate the secondary table to guarantee **exactly 1 row per join key** before joining:

In [ ]:
%%sql
-- CTE 2: Pre-aggregate daily Forex table to 1 row per date
WITH fx_daily AS (
    SELECT 
        DATE(rate_date) AS rate_date,
        AVG((CAST(ask_rate AS REAL) - CAST(bid_rate AS REAL)) / CAST(spot_rate AS REAL)) AS daily_spread_margin
    FROM fx_rates
    GROUP BY DATE(rate_date)
)
SELECT * FROM fx_daily LIMIT 3;

### 🔹 Substep 3.4 & 3.5: Relational Join & Window Functions (CTE 3)
Join the cleaned tables and apply row-level arithmetic or window analytics (e.g., `ROW_NUMBER()`, `LAG()`, `SUM() OVER`):

In [ ]:
%%sql
-- CTE 3: Join transactions with customers & fx rates
WITH valid_tx AS (
    SELECT 
        CAST(customer_id AS INTEGER) AS customer_id,
        DATE(transaction_date) AS tx_date,
        CAST(transaction_amount AS REAL) AS tx_amount,
        CAST(is_fraud AS INTEGER) AS is_fraud,
        TRIM(region) AS region
    FROM transactions
    WHERE LOWER(TRIM(transaction_status)) = 'completed'
),
fx_daily AS (
    SELECT 
        DATE(rate_date) AS rate_date,
        AVG((CAST(ask_rate AS REAL) - CAST(bid_rate AS REAL)) / CAST(spot_rate AS REAL)) AS daily_spread_margin
    FROM fx_rates
    GROUP BY DATE(rate_date)
),
enriched_tx AS (
    SELECT 
        t.customer_id,
        t.region,
        c.account_tier,
        t.tx_amount,
        t.is_fraud,
        t.tx_amount * COALESCE(fx.daily_spread_margin, 0.00124) AS fx_revenue_usd
    FROM valid_tx t
    INNER JOIN customers c ON t.customer_id = c.customer_id
    LEFT JOIN fx_daily fx ON t.tx_date = fx.rate_date
)
SELECT * FROM enriched_tx LIMIT 3;

### 🔹 Substep 3.6 & 3.7: Group by Target Grain, Conditional Aggregation & Sort (Final SELECT)
Collapse rows into the Step 1 target grain, derive KPIs with zero-division safety, and sort top results:

In [ ]:
%%sql
-- Complete Modular SQL Query: Target Grain = 1 row per account_tier
WITH valid_tx AS (
    SELECT 
        CAST(customer_id AS INTEGER) AS customer_id,
        DATE(transaction_date) AS tx_date,
        CAST(transaction_amount AS REAL) AS tx_amount,
        CAST(is_fraud AS INTEGER) AS is_fraud,
        TRIM(region) AS region
    FROM transactions
    WHERE LOWER(TRIM(transaction_status)) = 'completed'
),
fx_daily AS (
    SELECT 
        DATE(rate_date) AS rate_date,
        AVG((CAST(ask_rate AS REAL) - CAST(bid_rate AS REAL)) / CAST(spot_rate AS REAL)) AS daily_spread_margin
    FROM fx_rates
    GROUP BY DATE(rate_date)
),
enriched_tx AS (
    SELECT 
        t.customer_id,
        t.region,
        c.account_tier,
        t.tx_amount,
        t.is_fraud,
        t.tx_amount * COALESCE(fx.daily_spread_margin, 0.00124) AS fx_revenue_usd
    FROM valid_tx t
    INNER JOIN customers c ON t.customer_id = c.customer_id
    LEFT JOIN fx_daily fx ON t.tx_date = fx.rate_date
)
SELECT 
    account_tier,
    COUNT(DISTINCT customer_id) AS total_customers,
    ROUND(SUM(tx_amount), 2) AS total_volume_usd,
    ROUND(SUM(CASE WHEN is_fraud = 1 THEN tx_amount ELSE 0 END), 2) AS fraud_loss_usd,
    ROUND(SUM(fx_revenue_usd), 2) AS total_fx_revenue_usd,
    ROUND((SUM(CASE WHEN is_fraud = 1 THEN tx_amount ELSE 0 END) / NULLIF(SUM(tx_amount), 0)) * 100, 4) AS fraud_loss_rate_pct
FROM enriched_tx
GROUP BY account_tier
ORDER BY total_volume_usd DESC;

---
### ⚡ Alternative Approach: Fast 2-Block Interview SQL Pattern

> **Core Idea:** In live 20-30 minute coding interviews, you can write production SQL in **2 high-speed blocks** without losing any modularity:

<pre style="background: transparent !important; background-color: transparent !important; border: none !important; font-family: 'Courier New', Courier, monospace; font-size: 13px; line-height: 1.25; color: inherit; padding: 0; margin: 15px 0;">
┌────────────────────────────────────────────────────────┐
│ 📦 BLOCK 1: STAGING CTE (Substeps 3.1 + 3.2 + 3.3)     │
│ Clean primary table & pre-aggregate auxiliary tables   │
├────────────────────────────────────────────────────────┤
│ ⛓️ BLOCK 2: JOIN & AGGREGATE (Substeps 3.4 to 3.7)     │
│ Join staging tables, GROUP BY target grain, calculate  │
│ KPIs, and ORDER BY top results in a single SELECT      │
└────────────────────────────────────────────────────────┘
</pre>

#### 🎯 Can this be used for all types & levels of questions?
**Yes, 100%.** Here is how the exact same 2-block structure handles every level from Junior to Principal:

| Difficulty Level | How the 2-Block Combination Works |
| :--- | :--- |
| 🟢 **Junior / Entry Level**<br>*(Single table aggregations)* | **Block 1:** Clean & filter fact table in 1 CTE.<br>**Block 2:** `SELECT ... GROUP BY ... ORDER BY` *(Takes 60 seconds)*. |
| 🟡 **Mid-Level**<br>*(Multi-Table Merges, Ratios)* | **Block 1:** Staging CTE with pre-aggregated rates/disputes.<br>**Block 2:** `LEFT JOIN` ➔ `GROUP BY` ➔ `SUM(CASE WHEN ...)` *(Takes 3 mins)*. |
| 🔴 **Senior / Staff Level**<br>*(Window Analytics, Streaming ATO)* | **Block 1:** CTE with `LAG()` / `ROW_NUMBER()` time-deltas.<br>**Block 2:** `GROUP BY account_tier` ➔ Composite risk index *(Takes 5 mins)*. |

---
---

# 🛡️ STEP 4: DEFENSIVE SQL & SANITY VERIFICATION (The Guarantee)
> **Core Idea:** Senior engineers always build defensive boundaries into their SQL to guarantee zero crashes.

### 🔹 Substep 4.1: The Zero-Division Defense (`NULLIF`)
If a merchant or tier has 0 transaction volume, `fraud_loss / total_volume` causes a fatal division-by-zero crash.
- **SQL Shield:** Wrap denominators with `NULLIF(col, 0)`. If the volume is 0, SQL returns `NULL` instead of crashing:

In [ ]:
%%sql
-- Safe Division: Returns NULL instead of crashing when denominator is 0
SELECT 
    region,
    SUM(transaction_amount) AS volume,
    ROUND(SUM(is_fraud) * 100.0 / NULLIF(COUNT(*), 0), 2) AS fraud_rate_pct
FROM transactions
GROUP BY region;

### 🔹 Substep 4.2: Missing Value / NULL Defense (`COALESCE`)
When doing a `LEFT JOIN`, missing records return `NULL`. In SQL, `100 * NULL = NULL`.
- **SQL Shield:** Wrap joined values with `COALESCE(joined_col, fallback_value)` to ensure smooth calculations.

In [ ]:
%%sql
-- COALESCE prevents NULL contamination in financial sums
SELECT 
    t.customer_id,
    t.transaction_amount,
    COALESCE(c.account_tier, 'STANDARD') AS safe_account_tier
FROM transactions t
LEFT JOIN customers c ON t.customer_id = c.customer_id
LIMIT 5;

### 🔹 Substep 4.3: Target Grain Uniqueness Check
Prove that your final `GROUP BY` produced exactly 1 row per unique key:

In [ ]:
%%sql
-- Sanity Audit: Check that output grain has ZERO duplicates
WITH output_table AS (
    SELECT 
        account_tier,
        SUM(transaction_amount) AS total_volume
    FROM customers c
    JOIN transactions t ON c.customer_id = t.customer_id
    GROUP BY account_tier
)
SELECT 
    COUNT(*) AS total_output_rows,
    COUNT(DISTINCT account_tier) AS unique_grain_keys,
    CASE 
        WHEN COUNT(*) = COUNT(DISTINCT account_tier) THEN '✅ PASSED: Grain is 100% Unique'
        ELSE '❌ FAILED: Duplicate Grain Keys Detected'
    END AS audit_status
FROM output_table;

---
---

# 💡 Quick Framework Comparison: Pandas vs. SQL

| Step | Mental Question | Pandas Syntax | SQL Syntax |
| :--- | :--- | :--- | :--- |
| **Step 1 (Output)** | *"What does the target table look like?"* | Columns, 1-Row Grain, Shape | Final `SELECT` Columns, Target Grain, `LIMIT` |
| **Step 2 (Audit)** | *"What raw tables & quirks exist?"* | `df.info()`, `.nunique()`, nulls | `PRAGMA table_info`, `COUNT(DISTINCT)` |
| **Step 3.1** | *"How to cast & clean strings?"* | `pd.to_datetime()`, `.astype()`, `.str.strip()` | `CAST(... AS REAL)`, `DATE()`, `TRIM()` |
| **Step 3.2** | *"How to filter bad rows early?"* | `df[mask]` or `.query()` | `WHERE condition` in CTE 1 |
| **Step 3.3** | *"How to avoid 1:N duplicate rows?"* | Pre-aggregate child table `.groupby()` | CTE 2: `GROUP BY join_key` before joining |
| **Step 3.4** | *"How to join tables safely?"* | `.merge(how='left')` | `LEFT JOIN ... ON ...` in CTE 3 |
| **Step 3.5** | *"How to compute window analytics?"* | `.shift()`, `.transform()`, `.rolling()` | `OVER (PARTITION BY ... ORDER BY ...)` |
| **Step 3.6** | *"How to aggregate to target grain?"* | `.groupby().agg()` | Final `SELECT ... GROUP BY ... HAVING ...` |
| **Step 4** | *"How to defend against errors?"* | `assert`, `.fillna()`, bounds | `NULLIF(denom, 0)`, `COALESCE()`, Grain Check |